# 01 — Inspect the data
Inspect the actual CSV before writing model code. The guide's contract expects a timestamp, merchant_id and device_id; this dataset has none of them.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config

In [ ]:
raw = pd.read_csv(config.RAW_PATH)
print("shape:", raw.shape)
print(raw.dtypes)
print(raw.isna().mean().sort_values(ascending=False).head())
raw.head()

In [ ]:
from src.schema import resolve_columns, to_internal_schema, data_quality_report, UNAVAILABLE_IN_DATASET

print(resolve_columns(raw))
print("not available:", UNAVAILABLE_IN_DATASET)
transactions = to_internal_schema(raw)
data_quality_report(transactions)

## Target distribution
About 7.8% fraud. Accuracy would be misleading: predicting 'legitimate' for everything scores 92%.

In [ ]:
print(transactions["is_fraud"].value_counts())
print("fraud rate:", transactions["is_fraud"].mean())

## What drives the synthetic label?
Fraud concentrates in hours 1–4 combined with velocity ≥ 6. Amount, country, device and category carry no signal. Row order carries no time trend.

In [ ]:
print(transactions.groupby("hour")["is_fraud"].mean().round(3).to_dict())
print(transactions.groupby("velocity_1h")["is_fraud"].mean().round(3).to_dict())
for column in ["country", "device", "merchant_category"]:
    print(column, transactions.groupby(column)["is_fraud"].mean().round(3).to_dict())
night, fast = transactions["hour"].between(1, 4), transactions["velocity_1h"] >= 6
pd.crosstab([night.rename("hour 1-4"), fast.rename("velocity>=6")], transactions["is_fraud"], margins=True)

In [ ]:
order_decile = pd.qcut(range(len(transactions)), 10, labels=False)
transactions.groupby(order_decile)["is_fraud"].mean().round(3).tolist()  # flat: row order is not time